In [34]:
%matplotlib widget

import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import random
import math
import operator

In [35]:
df = pd.read_csv("iris.csv")
display(df.head())
display(df.shape)

FEATURES = df.columns.values.tolist()
PREDICT = FEATURES.pop()
display(FEATURES)

,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


(150, 5)

['sepal.length', 'sepal.width', 'petal.length', 'petal.width']

In [36]:
# get 70% train and 30% test
msk = np.random.rand(150) < 0.7     # display(msk)
train = df[msk]
test = df[~msk]
display(test.shape)

(37, 5)

## **Decision Tree**

In [37]:
class Node():
    def __init__(self, l=None, r=None, label=None, m=None):
        self.median = m
        self.left = l
        self.right = r
        self.label = label

    def leaf(self):
        return self.left==None and self.right==None

In [38]:
def get_split(data, feat):
    md = data[feat].median()
    left = data[data[feat] <= md]
    right = data[data[feat] > md]
    # print(left, right)
    return left, right

def get_frequency(data, feature):
    return len(data[feature]), dict(data[feature].value_counts())

def get_entropy(data, feat):
    ln, classes = get_frequency(data, feat)
    return -np.sum([v/ln * np.log2(v/ln) for k, v in classes.items()])

def get_info_gain(data):
    ln = len(data[PREDICT])
    global_entropy = get_entropy(data, PREDICT)
    weighted_entropy = 0
    info = {}

    for f in FEATURES:
        l_group, r_group = get_split(data, f)
        weighted_entropy = len(l_group)/ln * get_entropy(l_group, PREDICT) + len(r_group)/ln * get_entropy(r_group, PREDICT)
        info[f] =  global_entropy - weighted_entropy
        # print(f'{f} : {global_entropy} - {weighted_entropy} = {global_entropy - weighted_entropy}')
    # print(info)
        
    # getting key(feature) with maximum value(info_gain) in dictionary
    return max(info.items(), key=operator.itemgetter(1))[0]     

def get_gini():
    pass

def printTree(node, level=0):
    if node != None:
        printTree(node.left, level + 1)
        print(' ' * 4 * level + '-> ' + node.label)
        printTree(node.right, level + 1)


In [39]:
class Tree():
    def __init__(self):
        self.root = None
    
    def build(self, data):
        # get classes of current node
        classes = np.unique(data[PREDICT])     
        
        # stopping criteria
        if len(classes) == 1:
            # return leaf node because data is pure
                # print(f'\tc: {classes}')
            return Node(label=classes[0])       
        else:
            # select attribute B according heuristic_function
            best_feature = get_info_gain(data)     
                # print(f'\t{classes} \t{best_feature}')

            # generate left and right nodes until they become pure
            l_group, r_group = get_split(data, best_feature)
                # print("L")
            left_tree = self.build(l_group)
                # print("R")
            right_tree = self.build(r_group)
            return Node(l=left_tree, r=right_tree, label=best_feature, m=data[best_feature].median())
    
    def train(self, data):
        self.root = self.build(data)
    
    def test(self, data):
        bool_list = [self.traversal(self.root, d)==d[PREDICT] for d in data.to_dict(orient='records')]
        return 100*sum(bool_list)/len(bool_list)
        
    def traversal(self, node, row):
        if node.leaf():
            return node.label
        elif row[node.label] <= node.median:
            return self.traversal(node.left, row)
        else:
            return self.traversal(node.right, row)


In [40]:
t = Tree()
t.train(train)
print(f'{round(t.test(test), 5)}% accuracy')

91.89189% accuracy


In [41]:
print("-- DECISION TREE --\n")
printTree(t.root)

-- DECISION TREE --

        -> Setosa
    -> petal.length
            -> Versicolor
        -> sepal.width
                -> Versicolor
            -> sepal.width
                    -> Setosa
                -> sepal.length
                        -> Setosa
                    -> sepal.length
                        -> Versicolor
-> petal.length
            -> Versicolor
        -> petal.width
                -> Virginica
            -> sepal.length
                    -> Virginica
                -> sepal.length
                            -> Versicolor
                        -> petal.length
                            -> Virginica
                    -> sepal.length
                        -> Virginica
    -> petal.length
        -> Virginica


### **References**
- [probabilidad condicional](https://es.wikipedia.org/wiki/Probabilidad_condicionada)
- [media mediana y varianza]()
- [bagging bootstrap aggregating]()
- [bias in ML](https://www.bmc.com/blogs/bias-variance-machine-learning/)
- [decision tree sklearn](https://scikit-learn.org/stable/auto_examples/tree/plot_iris_dtc.html)
- [get list from pd.Df column or row](https://stackoverflow.com/questions/22341271/get-list-from-pandas-dataframe-column-or-row)
- [iterate over all or certain](https://thispointer.com/pandas-loop-or-iterate-over-all-or-certain-columns-of-a-dataframe/)